In [11]:
from langchain_core.documents import Document

In [ ]:
doc=Document(page_content="This is the content for RAG",
 metadata={
    "source": "example.com",
    "author": "John Doe",
    "date": "2021-01-01"
    })

print(doc.page_content)
print(doc.metadata)

In [ ]:
### Text Loader

from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/textfiles/ishan.txt")
docs = loader.load()
print(docs)




In [ ]:
### PDF Loader
from langchain_community.document_loaders import PyMuPDFLoader
loader=PyMuPDFLoader("../data/pdf/fire_alarm.pdf")  
docs=loader.load()
print(docs)

In [ ]:
##Directory Loader for TExt Files
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader("../data/textfiles",
glob="*.txt",
loader_cls=TextLoader,
loader_kwargs={"encoding":"utf-8"}
)
docs=dir_loader.load()
print(docs)

In [ ]:
##Directory Loader for TExt Files
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader("../data/pdf",
glob="*.pdf",
loader_cls=PyMuPDFLoader,
)
docs=dir_loader.load()
print(docs)

In [ ]:
### PDF Chunking helpers
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def load_pdf(path: str) -> list[Document]:
    """Load a single PDF into one Document per page."""
    return PyMuPDFLoader(path).load()


def load_pdf_dir(dir_path: str, glob: str = "*.pdf") -> list[Document]:
    """Load every PDF in a directory."""
    return DirectoryLoader(dir_path, glob=glob, loader_cls=PyMuPDFLoader).load()


def chunk_documents(
    docs: list[Document],
    chunk_size: int = 800,
    chunk_overlap: int = 100,
) -> list[Document]:
    """Split loaded documents into overlapping chunks for RAG.

    Metadata from the source page is preserved on every chunk, and a
    `chunk_id` is added so chunks stay traceable back to their document.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
    return chunks


def chunk_pdf(
    path: str,
    chunk_size: int = 800,
    chunk_overlap: int = 100,
) -> list[Document]:
    """Load a single PDF and return its chunks."""
    return chunk_documents(load_pdf(path), chunk_size, chunk_overlap)


def chunk_pdf_dir(
    dir_path: str,
    chunk_size: int = 800,
    chunk_overlap: int = 100,
) -> list[Document]:
    """Load all PDFs in a directory and return their chunks."""
    return chunk_documents(load_pdf_dir(dir_path), chunk_size, chunk_overlap)

In [ ]:
### Try the chunker on the fire alarm PDF
chunks = chunk_pdf("../data/pdf/lesson.pdf", chunk_size=500, chunk_overlap=80)

print(f"Total chunks: {len(chunks)}")
for c in chunks[:3]:
    print("-" * 60)
    print("page:", c.metadata.get("page"), "| chunk_id:", c.metadata.get("chunk_id"))
    print("chars:", len(c.page_content))
    print(c.page_content[:200], "...")

In [14]:
### Open-source embeddings + ChromaDB storage
from pathlib import Path

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

DEFAULT_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DEFAULT_CHROMA_DIR = "../data/chroma_db"
DEFAULT_COLLECTION = "portfolio"


def get_embeddings(model_name: str = DEFAULT_EMBEDDING_MODEL) -> HuggingFaceEmbeddings:
    """Create a local open-source embedding model (runs on CPU, no API key)."""
    return HuggingFaceEmbeddings(model_name=model_name)


def create_vector_store(
    chunks: list[Document],
    persist_directory: str = DEFAULT_CHROMA_DIR,
    collection_name: str = DEFAULT_COLLECTION,
    model_name: str = DEFAULT_EMBEDDING_MODEL,
) -> Chroma:
    """Embed chunks and store them in a persistent ChromaDB collection."""
    Path(persist_directory).mkdir(parents=True, exist_ok=True)
    embeddings = get_embeddings(model_name)

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name,
    )


def load_vector_store(
    persist_directory: str = DEFAULT_CHROMA_DIR,
    collection_name: str = DEFAULT_COLLECTION,
    model_name: str = DEFAULT_EMBEDDING_MODEL,
) -> Chroma:
    """Load an existing ChromaDB collection from disk."""
    return Chroma(
        persist_directory=persist_directory,
        collection_name=collection_name,
        embedding_function=get_embeddings(model_name),
    )


def search_vector_store(
    vector_store: Chroma,
    query: str,
    k: int = 3,
) -> list[Document]:
    """Return the top-k most similar chunks for a query."""
    return vector_store.similarity_search(query, k=k)

In [ ]:
### Embed PDF chunks and store in ChromaDB
pdf_chunks = chunk_pdf("../data/pdf/lesson.pdf", chunk_size=500, chunk_overlap=80)

vector_store = create_vector_store(
    chunks=pdf_chunks,
    persist_directory="../data/chroma_db",
    collection_name="portfolio",
)

print(f"Stored {vector_store._collection.count()} chunks in ChromaDB")

In [ ]:
### Test retrieval from ChromaDB
query = " What is the Assignment?"
results = search_vector_store(vector_store, query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print("page:", doc.metadata.get("page"), "| chunk_id:", doc.metadata.get("chunk_id"))
    print(doc.page_content[:500], "...\n")

In [ ]:
### RAG answers with Groq 
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

#  llama-3.1-8b-instant, openai/gpt-oss-20b
DEFAULT_GROQ_MODEL = "llama-3.1-8b-instant"
ENV_PATH = Path("../.env")


def load_env(env_path: Path = ENV_PATH) -> None:
    """Load API keys from a .env file."""
    if not load_dotenv(env_path):
        raise FileNotFoundError(
            f"Missing {env_path}. API"
        )

    if not os.getenv("GROQ_API_KEY"):
        raise ValueError(
            "No Groq API found"

        )


def get_llm(model: str = DEFAULT_GROQ_MODEL) -> ChatGroq:
    """Create a Groq chat model using the API key from .env."""
    load_env()
    return ChatGroq(
        model=model,
        api_key=os.environ["GROQ_API_KEY"],
        temperature=0.2,
        max_retries=3,
    )

def format_docs(docs: list) -> str:
    """Join retrieved chunks into one context string for the prompt."""
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


def build_rag_chain(vector_store: Chroma, k: int = 3):
    """Build a retrieval + LLM chain over an existing Chroma collection."""
    retriever = vector_store.as_retriever(search_kwargs={"k": k})

    prompt = ChatPromptTemplate.from_template(
        """You are a helpful assistant for a portfolio knowledge base.
Answer the user's question using ONLY the context below.
If the answer is not in the context, say: "I don't have enough information in the provided documents to answer that."

Context:
{context}

Question: {question}

Answer:"""
    )

    llm = get_llm()

    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )


def ask_rag(
    question: str,
    vector_store: Chroma,
    k: int = 3,
) -> str:
    """Ask a question and get an answer grounded in retrieved chunks."""
    chain = build_rag_chain(vector_store, k=k)
    return chain.invoke(question)

In [ ]:

import os

from groq import Groq

load_env()
for m in Groq(api_key=os.environ["GROQ_API_KEY"]).models.list().data:
    print(m.id)

In [10]:
### GROQ

try:
    vector_store
except NameError:
    vector_store = load_vector_store(
        persist_directory="../data/chroma_db",
        collection_name="portfolio",
    )

question = "Elaborate about Dining Philosopher Problem"
answer = ask_rag(question, vector_store, k=3)

print(f"Question: {question}\n")
print(f"Answer:\n{answer}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8176.50it/s]


Question: Elaborate about Dining Philosopher Problem

Answer:
The Dining Philosopher Problem is a classic problem in computer science that describes a scenario where five philosophers are seated around a circular table, each with a plate of spaghetti. The spaghetti is so slippery that a philosopher needs two forks to eat it, and there is one fork between each pair of plates.

The life of a philosopher consists of alternative periods of eating and thinking. When a philosopher gets hungry, they try to acquire their left and right forks. If they successfully acquire both forks, they eat for a while and then put down the forks to continue thinking. However, if all the philosophers take their left fork simultaneously, none will be able to take their right fork, resulting in a deadlock.
